# Gemma 4 Exploration Notebook

Covers: **Text · Hindi/Multilingual · Vision (Image) · Audio · Clinical Reasoning**

**How to run:** Upload to Google Colab or Kaggle. Set your `HF_TOKEN` in the secrets panel.

**Models tested:**
- `google/gemma-4-e2b-it` — 2B, text+image+audio, fast, free tier
- `google/gemma-4-e4b-it` — 4B, same modalities, better quality
- `google/gemma-4-26b-a4b-it` — 26B MoE (activates ~4B params), best clinical quality

**Two modes:**
- `InferenceClient` — calls HF serverless API, no GPU needed
- Local via `transformers` — runs on Colab T4/Kaggle P100; use for audio and deeper tests

> Note: Gemma 4 was released March 31 2026. Some features (audio via API, transformers audio pipeline) may still be stabilizing. Cells that may need updates are marked with ⚠️.

## 0. Setup

In [ ]:
!pip install -q huggingface_hub transformers accelerate Pillow requests bitsandbytes

In [ ]:
import os

# Colab: Secrets panel (key icon in sidebar) → add HF_TOKEN
# Kaggle: Add → Secrets → HF_TOKEN
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except ImportError:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except ImportError:
        HF_TOKEN = os.environ.get('HF_TOKEN', '')

assert HF_TOKEN, "Set HF_TOKEN in Colab/Kaggle secrets"

# Log in so transformers can pull gated weights
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("Authenticated")

In [ ]:
from huggingface_hub import InferenceClient

# Model IDs — swap freely
E2B = "google/gemma-4-e2b-it"
E4B = "google/gemma-4-e4b-it"
MoE_26B = "google/gemma-4-26b-a4b-it"

client = InferenceClient(token=HF_TOKEN)

def chat(messages, model=E2B, max_tokens=512, temperature=0.2):
    """Thin wrapper around InferenceClient chat completions."""
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content

def ask(prompt, model=E2B, **kwargs):
    """Single-turn text prompt."""
    return chat([{"role": "user", "content": prompt}], model=model, **kwargs)

print("Client ready")

---
## 1. Text: Basic Reasoning & Instruction Following

In [ ]:
# 1a. Simple factual — baseline sanity check
print(ask("What are the 3 most common warning signs of preeclampsia in pregnancy?"))

In [ ]:
# 1b. Structured output — does it follow JSON instructions?
prompt = """A pregnant woman, 28 weeks, BP 150/100, headache, swollen feet.
Respond ONLY as valid JSON with keys: risk_level (high/medium/low), 
reasons (list of strings), recommendation (string)."""

print(ask(prompt, model=E2B))

In [ ]:
# 1c. Multi-turn conversation
messages = [
    {"role": "user", "content": "A newborn is 2.1 kg at birth. Is that normal?"},
    {"role": "assistant", "content": "2.1 kg is below the normal range (2.5–4.0 kg), classifying this as low birth weight (LBW). The baby needs extra monitoring for temperature, feeding, and infection risk."},
    {"role": "user", "content": "What should the ASHA worker check first at home visit day 3?"}
]
print(chat(messages, model=E2B))

In [ ]:
# 1d. E2B vs 26B — same clinical prompt, compare quality
clinical_prompt = """An ASHA worker visits a 30-week pregnant woman.
Fundal height: 28 cm. Fetal movements: reduced (3-4 per hour). 
BP: 135/88. Hemoglobin: 9.2 g/dL.
List all risk flags, their severity, and the exact referral action."""

print("=== E2B ===")
print(ask(clinical_prompt, model=E2B, max_tokens=600))
print()
print("=== 26B A4B ===")
print(ask(clinical_prompt, model=MoE_26B, max_tokens=600))

---
## 2. Multilingual: Hindi

In [ ]:
# 2a. Hindi input, Hindi output
prompt_hi = """एक गर्भवती महिला 32 सप्ताह की है। उसका BP 145/95 है और सिरदर्द है।
आशा कार्यकर्ता को क्या करना चाहिए? हिंदी में जवाब दें।"""

print(ask(prompt_hi, model=E2B))

In [ ]:
# 2b. English prompt → Hindi response
print(ask(
    "Explain the 3 danger signs of postpartum hemorrhage. Reply in Hindi only.",
    model=E2B
))

In [ ]:
# 2c. Mixed-language (Hinglish) — common in rural India
print(ask(
    "Baby ka weight 2.0 kg hai aur usse theek se feed nahi ho raha. Kya karna chahiye?",
    model=E2B
))

In [ ]:
# 2d. JSON output in Hindi — does structure hold with non-Latin script?
prompt = """एक नवजात शिशु का वजन 1.9 किग्रा है और वह ठंडा और सुस्त है।
केवल JSON में उत्तर दें: {"risk_level": ..., "action": ..., "referral": true/false}"""

print(ask(prompt, model=E2B))

---
## 3. Vision: Image Understanding

Gemma 4 E2B/E4B support image inputs natively.

> ⚠️ Image support via `InferenceClient.chat.completions` for Gemma 4 may depend on HF serverless API rollout. If the cell errors, use the local transformers section below.

In [ ]:
# 3a. Image from URL — describe what you see
# Using a public medical chart image as example
# Replace IMAGE_URL with any publicly accessible image URL
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e9/Feetal_growth_restriction.jpg/320px-Feetal_growth_restriction.jpg"

messages = [{
    "role": "user",
    "content": [
        {"type": "image_url", "image_url": {"url": IMAGE_URL}},
        {"type": "text", "text": "Describe what you see in this image. Is there any clinical information visible?"}
    ]
}]

print(chat(messages, model=E2B))

In [ ]:
# 3b. Load a local image (e.g. a photo from your device)
# Upload a file to Colab first: Files panel → Upload
import base64
from pathlib import Path

def image_to_data_url(path):
    data = Path(path).read_bytes()
    b64 = base64.b64encode(data).decode()
    ext = Path(path).suffix.lstrip('.')
    return f"data:image/{ext};base64,{b64}"

# Uncomment and set path after uploading a file
# IMAGE_PATH = "/content/your_image.jpg"
# data_url = image_to_data_url(IMAGE_PATH)
# messages = [{
#     "role": "user",
#     "content": [
#         {"type": "image_url", "image_url": {"url": data_url}},
#         {"type": "text", "text": "What does this image show?"}
#     ]
# }]
# print(chat(messages, model=E2B))
print("Uncomment after uploading an image file")

In [ ]:
# 3c. Vision + Hindi — ask about an image in Hindi
messages = [{
    "role": "user",
    "content": [
        {"type": "image_url", "image_url": {"url": IMAGE_URL}},
        {"type": "text", "text": "इस छवि में क्या दिखाई दे रहा है? हिंदी में बताएं।"}
    ]
}]
print(chat(messages, model=E2B))

---
## 4. Audio

E2B and E4B support audio natively (26B/31B do not).

> ⚠️ Audio via HF serverless API for Gemma 4 may not be supported yet. The local transformers section below is the more reliable path for audio testing.

In [ ]:
# 4a. Audio via InferenceClient (may not work yet — depends on HF API rollout)
# Upload a .wav or .mp3 to Colab to test
import base64
from pathlib import Path

def audio_to_data_url(path):
    data = Path(path).read_bytes()
    b64 = base64.b64encode(data).decode()
    mime = "audio/wav" if path.endswith(".wav") else "audio/mpeg"
    return f"data:{mime};base64,{b64}"

# AUDIO_PATH = "/content/sample.wav"  # upload first
# audio_url = audio_to_data_url(AUDIO_PATH)
# messages = [{
#     "role": "user",
#     "content": [
#         {"type": "audio_url", "audio_url": {"url": audio_url}},
#         {"type": "text", "text": "Transcribe this audio and identify any medical terms mentioned."}
#     ]
# }]
# print(chat(messages, model=E2B))
print("Uncomment after uploading an audio file")

### 4b. Local audio via `transformers` (more reliable for audio)

Run this section on Colab/Kaggle with GPU runtime (T4 or better).

In [ ]:
# Check GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# Load E2B locally (requires ~4 GB VRAM in 4-bit, ~8 GB in bfloat16)
# ⚠️ Gemma 4 transformers class name may differ — check HF model card if AutoModel fails
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = E2B  # or E4B
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)

# 4-bit quantization to fit on T4 (16 GB)
from transformers import BitsAndBytesConfig
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=quant_config,
    device_map="auto",
)

print("Model loaded:", MODEL_ID)

In [ ]:
# Helper for local generation
def local_chat(messages, max_new_tokens=512):
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(DEVICE)
    
    input_len = inputs["input_ids"].shape[-1]
    
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    
    new_tokens = out[0][input_len:]
    return processor.decode(new_tokens, skip_special_tokens=True)

# Quick sanity check
print(local_chat([{"role": "user", "content": [{"type": "text", "text": "What is normal fetal heart rate?"}]}]))

In [ ]:
# 4c. Local image test
import requests
from PIL import Image
from io import BytesIO

img_response = requests.get(IMAGE_URL)
image = Image.open(BytesIO(img_response.content)).convert("RGB")

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "Describe this image and any clinical information it contains."}
    ]
}]

print(local_chat(messages))

In [ ]:
# 4d. Local audio test
# ⚠️ Audio input in Gemma 4 transformers may require a special audio processor
# This cell shows the expected API — may need adjustments as the library stabilizes

# AUDIO_PATH = "/content/sample.wav"
# import soundfile as sf  # !pip install soundfile
# audio_array, sample_rate = sf.read(AUDIO_PATH)

# messages = [{
#     "role": "user",
#     "content": [
#         {"type": "audio", "audio": audio_array},
#         {"type": "text", "text": "Transcribe this audio. Then list any medical terms."}
#     ]
# }]

# Note: the processor may need sample_rate passed separately:
# inputs = processor(audio=audio_array, sampling_rate=sample_rate, ...)
# Check the Gemma 4 HF model card for the exact API once it stabilizes.

print("Upload audio to /content/ and uncomment above")

---
## 5. Clinical Reasoning: Sakhi-Relevant Scenarios

These test the model on real ASHA workflows — the same prompts Sakhi sends to the backend.

In [ ]:
# 5a. High-risk ANC — should produce HIGH risk + PHC referral
anc_high = """Patient: Priya, 24 years, 34 weeks pregnant.
BP: 160/105. Protein in urine: +++. Severe headache and blurred vision.
Previous pregnancy ended in stillbirth.

As a clinical decision support tool for an ASHA worker:
1. Risk level (HIGH / MEDIUM / LOW)
2. Top 3 clinical concerns
3. Immediate action for the ASHA worker
4. What to tell the patient in simple language"""

print("=== E2B ===")
print(ask(anc_high, model=E2B))
print()
print("=== 26B A4B ===")
print(ask(anc_high, model=MoE_26B))

In [ ]:
# 5b. Same scenario in Hindi — critical for Sakhi
anc_high_hindi = """मरीज: प्रिया, 24 वर्ष, 34 सप्ताह गर्भवती।
BP: 160/105। पेशाब में प्रोटीन: +++। तेज सिरदर्द और धुंधली दृष्टि।
पिछली गर्भावस्था में मृत शिशु जन्म हुआ था।

आशा कार्यकर्ता के लिए नैदानिक निर्णय समर्थन:
1. जोखिम स्तर (उच्च / मध्यम / सामान्य)
2. मुख्य 3 चिंताएं
3. आशा के लिए तुरंत करने योग्य कदम
4. मरीज को सरल भाषा में क्या बताएं"""

print(ask(anc_high_hindi, model=E2B))

In [ ]:
# 5c. Newborn assessment — HBNC scenario
newborn = """Newborn, day 3. Weight: 1.8 kg (birth weight: 1.9 kg).
Temperature: 35.8°C. Not feeding well — latching but falling asleep after 2 min.
Jaundice visible below the knees.

Risk level, key concerns, and ASHA action plan."""

print(ask(newborn, model=E2B))

In [ ]:
# 5d. Hallucination probe — ask something with no right answer
# A good model should express uncertainty, not fabricate
print(ask(
    "What is the exact hemoglobin cutoff used by MOHFW India for severe anemia in pregnancy in the 2024 updated ANC guidelines?",
    model=E2B
))

---
## 6. Context Window: Long-Input Handling

E2B supports 128K tokens. Test whether it actually uses long context vs. just truncating.

In [ ]:
# Simulate a long patient history (10 ANC visits)
visits = []
for i in range(1, 11):
    week = 12 + (i - 1) * 3
    bp = f"{110 + i * 4}/{70 + i * 3}"
    hb = round(11.5 - i * 0.15, 1)
    visits.append(f"Visit {i} (Week {week}): BP {bp}, Hb {hb} g/dL, fundal height {week - 2} cm, no complaints")

# Sneak a critical finding into visit 7 (easy to miss)
visits[6] = visits[6].replace("no complaints", "proteinuria ++, pedal edema noted")

history = "\n".join(visits)
prompt = f"""Patient antenatal visit history:\n{history}\n\nWhich visit had the most concerning finding, and what should the ASHA have done at that visit?"""

print(ask(prompt, model=E2B))

---
## 7. System Prompt Adherence

Sakhi uses a system prompt that constrains format and adds clinical guardrails. Test if Gemma 4 respects it.

In [ ]:
system_prompt = """You are Sakhi, a clinical decision support assistant for ASHA workers in rural India.
Rules:
- Never diagnose. Always say 'Sakhi supports your judgment — always refer when unsure.'
- For HIGH or MEDIUM risk, always recommend PHC referral.
- Respond in the same language as the user.
- Be concise. Use simple language an ASHA worker can act on.
- Return structured JSON: {risk_level, flags, recommendation, refer_to_phc}"""

user_msg = "Patient, 28 weeks, BP 148/94, severe headache, first pregnancy."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_msg}
]

print("=== E2B with system prompt ===")
print(chat(messages, model=E2B))

In [ ]:
# Same system prompt, Hindi user message — does it switch language?
messages_hi = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "मरीज 28 सप्ताह, BP 148/94, तेज सिरदर्द, पहली गर्भावस्था।"}
]

print(chat(messages_hi, model=E2B))

---
## 8. Quick Scorecard

After running the cells, fill this in manually based on what you observed.

In [ ]:
scorecard = {
    "text_reasoning": "?/5",       # 1d: E2B vs 26B quality gap
    "json_format_adherence": "?/5",  # 1b: does it return clean JSON?
    "hindi_quality": "?/5",          # 2a-2d: fluency, clinical accuracy in Hindi
    "hinglish_handling": "?/5",      # 2c: natural handling of code-switching
    "vision_api": "works/fails",     # 3a: InferenceClient image
    "vision_local": "works/fails",   # 4c: local image
    "audio_api": "works/fails",      # 4a: InferenceClient audio
    "audio_local": "works/fails",    # 4d: local audio
    "clinical_risk_accuracy": "?/5",  # 5a: correct HIGH risk + PHC referral
    "hallucination_behavior": "?/5",  # 5d: expresses uncertainty vs. fabricates
    "long_context_recall": "?/5",     # 6: finds hidden finding in visit 7
    "system_prompt_adherence": "?/5", # 7: follows format + language switch
    "notes": ""
}

import json
print(json.dumps(scorecard, indent=2))